# Tech Stock Market Performance Analyzer
## ACC102 Mini-Assignment (Track 2)

### 1. Product Definition
This product analyzes the **market performance** of major technology companies  
using historical stock price data.

**Due to data availability, this version focuses on:**
- Price trends
- Return and risk metrics
- Comparative valuation (P/E ratio trend)

**This product solves thiis by:**
- **Input**: A list of company tickers (e.g., AAPL, MSFT) and a custom date range.
- **Process**: Automatically loading historical price data from local CSV files.
- **Output**: Visualizations of cumulative returns, volatility, and correlation analysis to aid investment decisions.

Users can modify:
- `TICKER_LIST`: Companies to compare
- `START_DATE` & `END_DATE`: Analysis window

## 2. Environment Setup: Install Required Libraries

Before proceeding with data loading, we ensure that the necessary Python libraries are installed.

Since we are working with **local CSV files** and performing **data analysis & visualization**, we require:

- `pandas` : For data manipulation and analysis.
- `matplotlib` : For plotting cumulative returns and other charts.
- `seaborn` : For enhanced statistical visualizations (e.g., correlation heatmap).

> Note: These libraries are typically pre-installed in Kaggle/Colab. This cell ensures compatibility.

In [1]:
# =====================================================
# SECTION: ENVIRONMENT SETUP
# Description: Install/verify libraries required for local data analysis.
# =========================================================

# Install required libraries (if not already available)
!pip install pandas matplotlib seaborn openpyxl

print("✅ Required libraries installed successfully.")

✅ Required libraries installed successfully.


## 3. Data Ingestion (Local CSV Files)
This section loads historical stock price data from local CSV files.
The file paths are defined using relative paths to ensure the notebook is portable and reproducible on GitHub.

In [20]:
# =========================================================
# SECTION 2: DATA LOADING FROM LOCAL FILES
# Description: Load historical stock data from CSV files.
# =========================================================

import pandas as pd
import os

# --- Step 1: Define relative file paths ---
# These paths assume the CSV files are in the same directory as the notebook
# or in a subdirectory named 'DATA'.
# IMPORTANT: Avoid hard-coding absolute paths like "C:\Users\..."
file_paths = {
    'AAPL': './DATA/aapl_historical_data.csv',
    'GOOGL': './DATA/GOOG_2004-08-19_2025-08-20.csv',
    'MSFT': './DATA/MSFT_Stock_Data_1986_2026.csv'
}

# --- Step 2: Load data into a dictionary ---
all_stock_data = {}

print("Loading local CSV files...")
for ticker, path in file_paths.items():
    try:
        # Check if file exists before reading
        if os.path.exists(path):
            df = pd.read_csv(path)
            # Standardize column names to lower case for consistency
            df.columns = df.columns.str.lower()
            all_stock_data[ticker] = df
            print(f"  ✅ Successfully loaded {ticker} from {path}")
        else:
            print(f"  ❌ ERROR: File not found for {ticker} at path: {path}")
    except Exception as e:
        print(f"  ❌ ERROR loading {ticker}: {e}")

print("\n✅ Data loading complete.")
print("Available tickers in memory:", list(all_stock_data.keys()))

Loading local CSV files...
  ✅ Successfully loaded AAPL from ./DATA/aapl_historical_data.csv
  ✅ Successfully loaded GOOGL from ./DATA/GOOG_2004-08-19_2025-08-20.csv
  ✅ Successfully loaded MSFT from ./DATA/MSFT_Stock_Data_1986_2026.csv

✅ Data loading complete.
Available tickers in memory: ['AAPL', 'GOOGL', 'MSFT']


## 4. Data Cleaning
This section processes the raw local data into analysis-ready format.

Key steps include:
1.  **Standardizing Column Names**: Converting all column headers to lowercase for consistency.
2.  **Date Parsing**: Ensuring the 'date' column is in datetime as the index.
3.  **Calculating Daily Returns**: Generating a new feature (`daily_return`) for performance analysis.

In [29]:
# =========================================================
# SECTION 4: DATA CLEANING & PREPROCESSING
# =========================================================
import pandas as pd

# Dictionary to hold cleaned data
cleaned_data = {}

print("Starting data cleaning process...")

for ticker, df in all_stock_data.items():
    print(f"\nProcessing data for {ticker}...")
    
   
    # Make a copy to avoid modifying the original data
    temp_df = df.copy()
    
    # --- Step 1: Standardize Column Names ---
    temp_df.columns = temp_df.columns.str.lower().str.strip()
    
    # Handle the issue of column names with suffixes for GOOGL (e.g., close googl → close)
    temp_df.columns = temp_df.columns.str.split().str[0]
    
    # --- Step 2: Handle Date Column ---
    date_col = None
    possible_date_cols = ['date', 'datetime', 'timestamp', 'time']
    for col in possible_date_cols:
        if col in temp_df.columns:
            date_col = col
            break
    
    if date_col is None:
        print(f"  ⚠️ WARNING: No date column found for {ticker}. Skipping this ticker.")
        continue
    
    # Rename to 'date'
    if date_col != 'date':
        temp_df.rename(columns={date_col: 'date'}, inplace=True)
    
    # Convert 'date' column to datetime
    temp_df['date'] = pd.to_datetime(temp_df['date'], errors='coerce')
    temp_df = temp_df.dropna(subset=['date'])
    
    # --- Step 3: Sort by Date ---
    temp_df = temp_df.sort_values('date').reset_index(drop=True)
    
    # --- Step 4: Handle Price Column ---
    price_col = None
    possible_price_cols = ['close', 'adj close', 'adjclose', 'price', 'last', 'adj_close', 'close/last']
    for col in possible_price_cols:
        if col in temp_df.columns:
            price_col = col
            break
    
    if price_col is None:
        print(f"  ⚠️ WARNING: No price column found for {ticker}. Skipping this ticker.")
        continue
    
    # Rename to 'price'
    if price_col != 'price':
        temp_df.rename(columns={price_col: 'price'}, inplace=True)
    
    # Remove dollar signs and convert to numeric (fix $ issue for AAPL)
    temp_df['price'] = temp_df['price'].astype(str).str.replace('$', '', regex=False)
    temp_df['price'] = pd.to_numeric(temp_df['price'], errors='coerce')
    temp_df = temp_df.dropna(subset=['price'])
    
    # --- Step 5: Calculate Daily Return ---
    temp_df['daily_return'] = temp_df['price'].pct_change()
    temp_df = temp_df.dropna()
    
    # --- Step 6: Store cleaned data ---
    cleaned_data[ticker] = temp_df
    print(f"  ✅ Cleaning complete for {ticker}. {len(temp_df)} rows remaining.")

print("\n✅ ALL DATA CLEANING COMPLETE!")
print("Available cleaned tickers:", list(cleaned_data.keys()))

Starting data cleaning process...

Processing data for AAPL...
  ✅ Cleaning complete for AAPL. 2512 rows remaining.

Processing data for GOOGL...
  ✅ Cleaning complete for GOOGL. 5283 rows remaining.

Processing data for MSFT...
  ✅ Cleaning complete for MSFT. 9873 rows remaining.

✅ ALL DATA CLEANING COMPLETE!
Available cleaned tickers: ['AAPL', 'GOOGL', 'MSFT']


## 5. Key Metrics Calculation
Calculate the three core financial indicators for each stock to measure performance and risk:
1. **Daily Return**: Daily percentage change in stock price
2. **Cumulative Return**: Total return over the entire period
3. **Annualized Volatility**: Risk level (higher = more volatile)

These metrics form the foundation of the risk‑return analysis.

In [30]:
# =========================================================
# SECTION 5: CALCULATE KEY ANALYSIS METRICS
# Calculate: Daily return | Cumulative return | Volatility
# =========================================================
import pandas as pd
import numpy as np

# Store final indicator results
metrics_result = []

# Iterate through the cleaned data of the 3 companies
for ticker, df in cleaned_data.items():
    print(f"\n📊 Calculating metrics for {ticker}...")

    # 1. Daily return (already calculated during cleaning, use directly)
    daily_return = df['daily_return']

    # 2. Cumulative return
    cumulative_return = (1 + daily_return).prod() - 1

    # 3. Volatility (annualized)
    volatility = daily_return.std() * np.sqrt(252)

    # Save results
    metrics_result.append({
        'Ticker': ticker,
        'Cumulative Return': cumulative_return,
        'Annualized Volatility': volatility
    })

# Convert results into a table
metrics_df = pd.DataFrame(metrics_result)

# Display final results
print("\n" + "="*60)
print("✅ ALL CALCULATIONS COMPLETED!")
print("="*60)
print(metrics_df.round(4))


📊 Calculating metrics for AAPL...

📊 Calculating metrics for GOOGL...

📊 Calculating metrics for MSFT...

✅ ALL CALCULATIONS COMPLETED!
  Ticker  Cumulative Return  Annualized Volatility
0   AAPL             8.4397                 0.2895
1  GOOGL            80.0241                 0.3066
2   MSFT          3917.7023                 0.3307


## 6. Generate Metrics Table for GitHub README
Convert calculated metrics into a clean, readable Markdown table.
This table will be used in the GitHub repository to summarize the performance of AAPL, GOOGL, and MSFT.

In [32]:
# =========================================================
# SECTION 6: EXPORT MARKDOWN TABLE FOR README
# =========================================================

# 1. Format data (keep 4 decimal places for better readability)
formatted_metrics = metrics_df.copy()
formatted_metrics['Cumulative Return'] = formatted_metrics['Cumulative Return'].round(4)
formatted_metrics['Annualized Volatility'] = formatted_metrics['Annualized Volatility'].round(4)

# 2. Generate Markdown table
print("Copy the following table to your GitHub README:\n")
print(formatted_metrics.to_markdown(index=False))

# 3. Save to file
with open("metrics_table.md", "w", encoding="utf-8") as f:
    f.write("# Stock Performance Metrics\n\n")
    f.write("## AAPL, GOOGL, MSFT Risk-Return Summary\n\n")
    f.write(formatted_metrics.to_markdown(index=False))
    f.write("\n\n")
    f.write("Generated using Python and Pandas.")

print("\n✅ Markdown table has been saved as metrics_table.md")

Copy the following table to your GitHub README:

| Ticker   |   Cumulative Return |   Annualized Volatility |
|:---------|--------------------:|------------------------:|
| AAPL     |              8.4397 |                  0.2895 |
| GOOGL    |             80.0241 |                  0.3066 |
| MSFT     |           3917.7    |                  0.3307 |

✅ Markdown table has been saved as metrics_table.md


## 7. Full Visualization
Create comprehensive visualizations to help interpret stock performance:
1. Stock price trend
2. Cumulative return comparison
3. Annualized volatility comparison
4. Daily return distribution
5. Rolling 30‑day volatility
6. Return correlation heatmap

Visuals make trends and risks easier to understand for investors and analysts.

In [33]:
# =========================================================
# SECTION 7: FULL VISUALIZATION (UPGRADED & FIXED VERSION)
# Generate six professional charts and fix the error of rolling volatility
# =========================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set plot style
plt.style.use('default')

# ====================== 1. Stock Price Trend Chart ======================
plt.figure(figsize=(12,5))
for ticker, df in cleaned_data.items():
    plt.plot(df['date'], df['price'], label=ticker)
plt.title('Stock Price Trend')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('1_price_trend.png')
plt.close()

# ====================== 2.Cumulative Return ======================
plt.figure(figsize=(12,5))
for ticker, df in cleaned_data.items():
    cr = (1 + df['daily_return']).cumprod() - 1
    plt.plot(df['date'], cr, label=ticker)
plt.title('Cumulative Return')
plt.xlabel('Date')
plt.ylabel('Return')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('2_cumulative_return.png')
plt.close()

# ====================== 3. Volatility Bar Chart. ======================
plt.figure(figsize=(8,5))
plt.bar(metrics_df['Ticker'], metrics_df['Annualized Volatility'])
plt.title('Annualized Volatility')
plt.tight_layout()
plt.savefig('3_volatility.png')
plt.close()

# ====================== 4. Histogram of Daily Returns Distribution ======================
plt.figure(figsize=(12,8))
for i, ticker in enumerate(cleaned_data.keys()):
    plt.subplot(3,1,i+1)
    plt.hist(cleaned_data[ticker]['daily_return'], bins=50, alpha=0.6, label=ticker)
    plt.title(f'{ticker} Daily Return Distribution')
    plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('4_return_distribution.png')
plt.close()

# ====================== 5. Rolling 30-Day Volatility======================
plt.figure(figsize=(12,5))
for ticker, df in cleaned_data.items():
    # Calculate rolling volatility
    vol_30 = df['daily_return'].rolling(30).std() * np.sqrt(252)
    # Remove NaN values (first 29 entries) and synchronize date slicing accordingly
    valid_mask = ~vol_30.isna()
    plt.plot(df['date'][valid_mask], vol_30[valid_mask], label=ticker)
plt.title('30-Day Rolling Volatility')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('5_rolling_volatility.png')
plt.close()

# ====================== 6. Correlation Heatmap ======================
# Align data for all three companies (use intersecting dates) to calculate correlation
aligned_dfs = []
for ticker, df in cleaned_data.items():
    temp = df[['date', 'daily_return']].rename(columns={'daily_return': ticker})
    aligned_dfs.append(temp)

# Merge data and align by dates
from functools import reduce
ret_df = reduce(lambda left, right: pd.merge(left, right, on='date', how='inner'), aligned_dfs)
ret_df = ret_df.drop('date', axis=1)

corr = ret_df.corr()
plt.figure(figsize=(6,5))
plt.imshow(corr, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns)
plt.yticks(range(len(corr)), corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j,i, round(corr.iloc[i,j],2), ha='center', va='center')

plt.title('Return Correlation Heatmap')
plt.tight_layout()
plt.savefig('6_correlation_heatmap.png')
plt.close()

# ====================== Output completed ======================
print("✅ All 6 charts have been successfully generated!")
print("1. 1_price_trend.png")
print("2. 2_cumulative_return.png")
print("3. 3_volatility.png")
print("4. 4_return_distribution.png")
print("5. 5_rolling_volatility.png")
print("6. 6_correlation_heatmap.png")

✅ All 6 charts have been successfully generated!
1. 1_price_trend.png
2. 2_cumulative_return.png
3. 3_volatility.png
4. 4_return_distribution.png
5. 5_rolling_volatility.png
6. 6_correlation_heatmap.png


## 8. Automated Investment Analysis & Recommendations
### Purpose:
Based on the calculated risk‑return metrics and visualized results,
this section automatically generates data-driven investment advice for different types of investors.
It transforms raw data into actionable insights, which is the core value of data analysis.

### Output:
- Performance summary of all stocks
- Tailored investment recommendations for growth-focused, risk-averse, and balanced investors
- Important risk disclosure

In [34]:
# =========================================================
# ## Auto-Generated Investment Recommendations
# Description: Generate tailored investment advice based on risk-return metrics
# =========================================================

# 1. Identify the best and worst performing metrics
top_return_ticker = metrics_df.loc[metrics_df['Cumulative Return'].idxmax(), 'Ticker']
lowest_return_ticker = metrics_df.loc[metrics_df['Cumulative Return'].idxmin(), 'Ticker']

top_vol_ticker = metrics_df.loc[metrics_df['Annualized Volatility'].idxmax(), 'Ticker']
lowest_vol_ticker = metrics_df.loc[metrics_df['Annualized Volatility'].idxmin(), 'Ticker']

# 2. Calculate risk-reward ratio
metrics_df['Risk-Return Ratio'] = metrics_df['Cumulative Return'] / metrics_df['Annualized Volatility']
best_risk_return_ticker = metrics_df.loc[metrics_df['Risk-Return Ratio'].idxmax(), 'Ticker']

# 3. Print analysis report
print("="*70)
print("📊 AUTOMATED INVESTMENT ANALYSIS & RECOMMENDATIONS")
print("="*70)
print(f"\n📈 Performance Summary:")
print(f"  - Highest Cumulative Return: {top_return_ticker} ({metrics_df['Cumulative Return'].max():.2f}x)")
print(f"  - Lowest Cumulative Return: {lowest_return_ticker} ({metrics_df['Cumulative Return'].min():.2f}x)")
print(f"  - Lowest Volatility (Most Stable): {lowest_vol_ticker} ({metrics_df['Annualized Volatility'].min():.2f})")
print(f"  - Highest Volatility (Riskiest): {top_vol_ticker} ({metrics_df['Annualized Volatility'].max():.2f})")

print(f"\n💡 Key Recommendations:")
print(f"  1. **For Growth-Focused Investors**: Prioritize {top_return_ticker}. It has delivered the highest long-term return, suitable for investors with a high risk tolerance.")
print(f"  2. **For Risk-Averse Investors**: Consider {lowest_vol_ticker}. It offers the most stable performance with the lowest volatility.")
print(f"  3. **Balanced Choice**: {best_risk_return_ticker} provides the best risk-return tradeoff, making it suitable for most investors.")

print(f"\n⚠️ Important Note: These recommendations are based solely on historical performance. Past returns do not guarantee future results.")
print("="*70)

📊 AUTOMATED INVESTMENT ANALYSIS & RECOMMENDATIONS

📈 Performance Summary:
  - Highest Cumulative Return: MSFT (3917.70x)
  - Lowest Cumulative Return: AAPL (8.44x)
  - Lowest Volatility (Most Stable): AAPL (0.29)
  - Highest Volatility (Riskiest): MSFT (0.33)

💡 Key Recommendations:
  1. **For Growth-Focused Investors**: Prioritize MSFT. It has delivered the highest long-term return, suitable for investors with a high risk tolerance.
  2. **For Risk-Averse Investors**: Consider AAPL. It offers the most stable performance with the lowest volatility.
  3. **Balanced Choice**: MSFT provides the best risk-return tradeoff, making it suitable for most investors.

⚠️ Important Note: These recommendations are based solely on historical performance. Past returns do not guarantee future results.
